# EDGAR Financial Sentiment — Part 8: The Bake-Off

**The convergence — and the portfolio centerpiece.** One results table scoring *every* approach you've built on a **single shared held-out set**: a lexicon baseline, fine-tuned GPT-2, BERT vs FinBERT, QLoRA Mistral, zero/few-shot LLM. The table tells the whole story — and the cost-vs-accuracy view is the decision the `Choosing_a_Training_Method` lab was preparing you for.

**What you'll practice (the core concept):** **fair benchmarking** — a single scoring **harness** (`record`) that every method funnels through, and a **lexicon baseline** (`lexicon_classify`). Those are your blanks; the per-method training is reused from earlier parts.

> **Run in Colab with a T4 GPU.** Self-contained — every row is produced live. The cheap methods (lexicon, GPT-2, BERT, FinBERT) run by default (~20 min). The heavy ones (QLoRA, LLM, EDGAR-pretrained GPT-2) are **flag-gated, off by default** — enabling all of them pushes runtime toward ~2 hours.

## 0. Why a bake-off? One table, one held-out set

You've built eight ways to classify a financial sentence. Scattered across eight notebooks, the numbers aren't comparable — different splits, different eval sizes, different days. A **bake-off** fixes that: **one held-out test set, one scoring function, every method run through it.** Only then can you say *X beats Y*.

Two things make this the centerpiece:
1. **Fair comparison is a skill.** The discipline is that nothing is special-cased — lexicon, a 7B QLoRA model, and a zero-shot prompt all produce predictions for the *same* sentences and get scored the *same* way. The harness enforces it.
2. **Accuracy isn't the only axis.** The table also carries a **cost** column. The lesson from the use-case lab lands here: the right model is the **cheapest one that clears your accuracy bar**, and you can only see that when accuracy and cost sit side by side.

**Memory reality:** a T4 can't hold these models at once, so each method **loads, trains, predicts, and frees the GPU** before the next runs. That sequencing is the whole reason this notebook is structured as one-method-at-a-time.

## 1. Setup

In [ ]:
!pip install -q -U transformers datasets peft bitsandbytes accelerate matplotlib

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import random, re, gc, io, zipfile, requests
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModel, AutoModelForCausalLM,
                          AutoModelForSequenceClassification, BitsAndBytesConfig)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset as HFDataset, ClassLabel
import matplotlib.pyplot as plt

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'Switch the Colab runtime to a T4 GPU before running.'

## 2. Config & flags

The heavy methods are **off by default**. Flip a flag to `True` to add that row (and its runtime). The Mistral methods need the gated-model HF login from Parts 4–7.

In [ ]:
torch.manual_seed(10); random.seed(10); np.random.seed(10)

MISTRAL = 'mistralai/Mistral-7B-Instruct-v0.3'
LABELS  = ['negative', 'neutral', 'positive']

# --- heavy-method flags (each adds significant runtime; needs Mistral login for the 7B ones) ---
RUN_QLORA      = False   # ~30 min: QLoRA fine-tune of Mistral-7B
RUN_LLM        = False   # ~30-45 min: zero- + few-shot generation over the full test set
RUN_GPT2_EDGAR = False   # ~12 min: continued-pretrain GPT-2 on EDGAR, then fine-tune (reproduces Parts 1-2)

if RUN_QLORA or RUN_LLM:
    from huggingface_hub import login; login()   # gated Mistral model

## 3. The shared held-out set

Same PhraseBank + `seed=10` split as every prior part, so this test set is **identical** to the one Parts 2–7 used. Every method below is scored on exactly these sentences.

In [ ]:
_URL = 'https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip'
_z = zipfile.ZipFile(io.BytesIO(requests.get(_URL, timeout=120).content))
_member = next(n for n in _z.namelist() if n.endswith('Sentences_AllAgree.txt'))
_raw = _z.read(_member).decode('latin-1')
_label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
_rows = []
for _line in _raw.splitlines():
    _line = _line.strip()
    if _line:
        _s, _l = _line.rsplit('@', 1); _rows.append({'sentence': _s, 'label': _label_map[_l.strip()]})
pb_full = HFDataset.from_pandas(pd.DataFrame(_rows)).cast_column('label', ClassLabel(names=LABELS))
pb = pb_full.train_test_split(test_size=0.2, seed=10, stratify_by_column='label')
pb_train, pb_test = pb['train'], pb['test']

train_pairs = [(e['sentence'], e['label']) for e in pb_train]
test_pairs  = [(e['sentence'], e['label']) for e in pb_test]
GOLD = [l for _, l in test_pairs]
print('Train:', len(train_pairs), '| Held-out test:', len(test_pairs))

RESULTS = []
def free():
    gc.collect(); torch.cuda.empty_cache()

## 4. The scoring harness  &larr; **your code**

`record(name, preds, cost)` is the heart of the bake-off: it takes a method's predictions (a list of class indices `0/1/2`, one per test sentence, in order), scores them against the shared `GOLD`, stores the row, and prints it. **Every** method — lexicon, BERT, QLoRA, LLM — goes through this one function, which is what makes the comparison fair.

In [ ]:
def record(name, preds, cost):
    """Score preds (list of 0/1/2, aligned with GOLD), store + print the row."""
    ### YOUR CODE HERE
    assert len(preds) == len(GOLD), f'{name}: got {len(preds)} preds for {len(GOLD)} test items'
    acc = 100 * sum(int(p == g) for p, g in zip(preds, GOLD)) / len(GOLD)
    RESULTS.append({'method': name, 'accuracy': acc, 'cost': cost})
    print(f'  {name:30s} {acc:5.1f}%   (cost: {cost})')
    return acc
    ### END YOUR CODE

## 5. Baseline — a Loughran-McDonald-style lexicon  &larr; **your code**

The cheapest possible method, and the historical baseline finance NLP started from: **count positive vs. negative finance words** and pick the winner. No training, no GPU. Loughran & McDonald (2011) showed generic sentiment lexicons fail on financial text ("liability" isn't negative in finance), so finance uses curated word lists — a small one is provided below. Write the classifier: count, compare, return `2` (positive), `0` (negative), or `1` (neutral, on a tie).

In [ ]:
POS_WORDS = {'gain','gains','growth','grew','rose','rise','beat','beats','strong','record','profit','profits',
             'increase','increased','higher','raised','improved','surged','rally','exceeded','robust','upbeat','up'}
NEG_WORDS = {'loss','losses','decline','declined','fell','drop','dropped','miss','missed','weak','weaker',
             'decrease','decreased','lower','lowered','cut','warning','negative','plunged','slump','charge',
             'restructuring','layoffs','down'}

def lexicon_classify(sentence):
    """Count POS vs NEG words; return 2 (pos), 0 (neg), or 1 (neutral on a tie)."""
    ### YOUR CODE HERE
    words = re.findall(r'[a-z]+', sentence.lower())
    pos = sum(w in POS_WORDS for w in words)
    neg = sum(w in NEG_WORDS for w in words)
    if pos > neg: return 2
    if neg > pos: return 0
    return 1
    ### END YOUR CODE

## 6. Predict the ranking *(before any model runs)*

#### ✅ A reasonable prediction
Roughly, best → worst accuracy: **FinBERT ≈ BERT-base** (mid-90s) > **QLoRA Mistral** (close, but a 7B isn't guaranteed to top a domain-matched FinBERT here) > **GPT-2 fine-tuned** (~80s) > **few-shot LLM > zero-shot LLM** (no training, ~70s–80s) > **lexicon** (the floor — word-counting can't read context, expect ~50–65%). On **cost** the order roughly inverts: lexicon and the LLM prompts need no training; the fine-tunes are mid; QLoRA trains few params but on a 7B. The headline you're testing: **the top is compressed and the cheapest decent method may already clear the bar.**

## 7. Run the cheap methods (always on)
Lexicon first (instant), then the live fine-tunes. Each trains, predicts on the shared test set, and frees the GPU before the next.

In [ ]:
# --- lexicon (no training) ---
record('Lexicon (LM-style)', [lexicon_classify(s) for s, _ in test_pairs], 'none (no training)')

In [ ]:
def train_and_predict(model_id, decoder=False, epochs=3, lr=2e-5, bs=16, train_examples=None):
    """Fine-tune a [CLS]/last-token classifier on train_examples, return preds for test_pairs, then free."""
    torch.manual_seed(10)
    tok = AutoTokenizer.from_pretrained(model_id)
    if decoder and tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left' if decoder else 'right'   # decoder: last token is the real one; encoder: [CLS] at 0
    enc = AutoModel.from_pretrained(model_id).to(device)
    pool = -1 if decoder else 0

    class _DS(Dataset):
        def __init__(self, ex):
            self.r = []
            for s, l in ex:
                e = tok(s, max_length=64, truncation=True, padding='max_length', return_tensors='pt')
                self.r.append({'input_ids': e['input_ids'].squeeze(0),
                               'attention_mask': e['attention_mask'].squeeze(0),
                               'label': torch.tensor(l, dtype=torch.int64)})
        def __len__(self): return len(self.r)
        def __getitem__(self, i): return self.r[i]

    class _Clf(nn.Module):
        def __init__(self, e):
            super().__init__(); self.e = e; self.h = nn.Linear(e.config.hidden_size, 3)
        def forward(self, b):
            o = self.e(input_ids=b['input_ids'], attention_mask=b['attention_mask'])
            return self.h(o.last_hidden_state[:, pool, :])

    tr = DataLoader(_DS(train_examples or train_pairs), batch_size=bs, shuffle=True)
    te = DataLoader(_DS(test_pairs), batch_size=32, shuffle=False)
    clf = _Clf(enc).to(device); opt = torch.optim.AdamW(clf.parameters(), lr=lr); lf = nn.CrossEntropyLoss()
    clf.train()
    for _ in range(epochs):
        for b in tr:
            b = {k: v.to(device) for k, v in b.items()}
            opt.zero_grad(); lf(clf(b), b['label']).backward(); opt.step()
    clf.eval(); preds = []
    with torch.no_grad():
        for b in te:
            b = {k: v.to(device) for k, v in b.items()}
            preds.extend(clf(b).argmax(-1).cpu().tolist())
    del clf, enc; free()
    return preds

In [ ]:
record('GPT-2 (fine-tuned)',     train_and_predict('gpt2', decoder=True), 'full FT ~124M')
record('BERT-base (fine-tuned)', train_and_predict('bert-base-cased'),    'full FT ~110M')
record('FinBERT (fine-tuned)',   train_and_predict('ProsusAI/finbert'),   'full FT ~110M')

## 8. Heavy methods (run only if their flag is on)
Each loads its model, runs, and frees the GPU. Skipped cleanly when the flag is `False`.

In [ ]:
def run_qlora(steps=500, lr=2e-4, bs=4):
    tok = AutoTokenizer.from_pretrained(MISTRAL); tok.pad_token = tok.eos_token; tok.padding_side = 'right'
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    m = AutoModelForSequenceClassification.from_pretrained(MISTRAL, num_labels=3,
                                                           quantization_config=bnb, device_map={'': 0})
    m.config.pad_token_id = tok.pad_token_id; m.config.use_cache = False
    m = prepare_model_for_kbit_training(m)
    m = get_peft_model(m, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                                     task_type=TaskType.SEQ_CLS,
                                     target_modules=['q_proj','k_proj','v_proj','o_proj']))
    def _enc(ex):
        e = tok([s for s,_ in ex], max_length=64, truncation=True, padding='max_length', return_tensors='pt')
        return e['input_ids'], e['attention_mask'], torch.tensor([l for _,l in ex])
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=lr); lf = nn.CrossEntropyLoss()
    m.train(); step = 0
    while step < steps:
        random.shuffle(train_pairs)
        for i in range(0, len(train_pairs) - bs, bs):
            ids, am, y = _enc(train_pairs[i:i+bs])
            ids, am, y = ids.to(device), am.to(device), y.to(device)
            opt.zero_grad(); lf(m(input_ids=ids, attention_mask=am).logits, y).backward(); opt.step()
            step += 1
            if step >= steps: break
    m.eval(); preds = []
    with torch.no_grad():
        for i in range(0, len(test_pairs), 8):
            ids, am, _ = _enc(test_pairs[i:i+8])
            preds.extend(m(input_ids=ids.to(device), attention_mask=am.to(device)).logits.argmax(-1).cpu().tolist())
    del m; free()
    return preds

if RUN_QLORA:
    print('Running QLoRA Mistral-7B (slow)...')
    record('QLoRA Mistral-7B', run_qlora(), 'LoRA <1% of 7B (4-bit)')

In [ ]:
def run_llm(shots=0):
    tok = AutoTokenizer.from_pretrained(MISTRAL)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    m = AutoModelForCausalLM.from_pretrained(MISTRAL, quantization_config=bnb, device_map={'': 0}); m.eval()
    examples = []
    if shots:
        for c in range(3):
            s = next(s for s, l in train_pairs if l == c); examples.append((s, LABELS[c]))
    L2I = {n: i for i, n in enumerate(LABELS)}
    def prompt(sentence):
        lines = ['Classify the sentiment of the financial sentence as exactly one of: '
                 'negative, neutral, positive. Reply with only that one word.', '']
        for s, lab in examples:
            lines += [f'Sentence: "{s}"', f'Sentiment: {lab}', '']
        lines += [f'Sentence: "{sentence}"', 'Sentiment:']
        return '\n'.join(lines)
    def parse(reply):
        t = reply.lower(); hits = [(t.find(w), i) for w, i in L2I.items() if w in t]
        return min(hits)[1] if hits else -1
    preds = []
    with torch.no_grad():
        for s, _ in test_pairs:
            inp = tok.apply_chat_template([{'role':'user','content':prompt(s)}],
                                          add_generation_prompt=True, return_tensors='pt').to(device)
            out = m.generate(inp, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)
            preds.append(parse(tok.decode(out[0][inp.shape[-1]:], skip_special_tokens=True)))
    del m; free()
    return preds

if RUN_LLM:
    print('Running zero-shot LLM (slow)...');  record('LLM zero-shot', run_llm(shots=0), 'none (prompt)')
    print('Running few-shot LLM (slow)...');   record('LLM few-shot',  run_llm(shots=1), 'none (prompt)')

In [ ]:
def run_gpt2_edgar(pretrain_steps=500, target_chunks=8000):
    # short continued pretraining of GPT-2 on EDGAR 10-K text (reproduces Part 1), then fine-tune (Part 2)
    tok = AutoTokenizer.from_pretrained('gpt2'); tok.add_special_tokens({'pad_token': '[PAD]'})
    from transformers import GPT2LMHeadModel
    lm = GPT2LMHeadModel.from_pretrained('gpt2').to(device); lm.resize_token_embeddings(len(tok))
    stream = __import__('datasets').load_dataset('eloukas/edgar-corpus', 'year_2020', split='train',
                                                 streaming=True, trust_remote_code=True)
    chunks = []
    for rec in stream:
        secs = [v for k, v in rec.items() if k.startswith('section_') and isinstance(v, str) and v.strip()]
        w = ' '.join(secs).split()
        chunks += [' '.join(w[i:i+120]) for i in range(0, len(w), 120)]
        if len(chunks) >= target_chunks: break
    enc = tok(chunks[:target_chunks], max_length=100, truncation=True, padding='max_length', return_tensors='pt')
    keep = enc['attention_mask'][:, 99] > 0; data = enc['input_ids'][keep].to(device)
    opt = torch.optim.AdamW(lm.parameters(), lr=1e-5); lf = nn.CrossEntropyLoss(); lm.train()
    for step in range(pretrain_steps):
        b = data[torch.randint(0, data.shape[0], (4,))]
        out = lm(b[:, :-1]).logits.reshape(-1, len(tok))
        opt.zero_grad(); lf(out, b[:, 1:].reshape(-1)).backward(); opt.step()
    lm.save_pretrained('/content/gpt2_edgar'); tok.save_pretrained('/content/gpt2_edgar')
    del lm, data; free()
    return train_and_predict('/content/gpt2_edgar', decoder=True)

if RUN_GPT2_EDGAR:
    print('Running EDGAR-pretrained GPT-2 (slow)...')
    record('GPT-2 EDGAR-pretrained', run_gpt2_edgar(), 'pretrain + full FT')

## 9. The results table

In [ ]:
rows = sorted(RESULTS, key=lambda r: r['accuracy'], reverse=True)
print(f'{"method":32s}{"accuracy":>10s}   cost')
print('-' * 70)
for r in rows:
    print(f'{r["method"]:32s}{r["accuracy"]:>9.1f}%   {r["cost"]}')

plt.figure(figsize=(8, 0.5 * len(rows) + 1))
order = sorted(RESULTS, key=lambda r: r['accuracy'])
plt.barh([r['method'] for r in order], [r['accuracy'] for r in order])
plt.xlabel('test accuracy (%)'); plt.xlim(0, 100)
plt.title('Bake-off: every method on the same held-out PhraseBank test set')
for i, r in enumerate(order):
    plt.text(r['accuracy'] + 1, i, f"{r['accuracy']:.1f}%", va='center')
plt.tight_layout(); plt.show()

## 10. Reflect — what the table says

#### ✅ The takeaways
- **The top is compressed.** BERT, FinBERT, and (if you ran it) QLoRA cluster in the mid-90s — once you're near the ceiling of a clean task, a 7B model barely beats a 110M domain-matched one. **Diminishing returns** are visible, not theoretical.
- **The lexicon is the floor, and that's the point.** Word-counting can't read context ("not strong" counts as positive), so it lands far below the trained models — which is *why* the field moved to learned representations. It cost nothing, though.
- **No-training prompting is a real contender.** The zero/few-shot LLM rows trail the fine-tunes but need **zero labels and zero training** — for many projects that tradeoff wins.
- **Read accuracy *and* cost together.** The decision the `Choosing_a_Training_Method` lab set up is now a table you can point at: pick the **cheapest method that clears your bar**. Often that's FinBERT (cheap, domain-matched, top-tier), not the 7B.

**This table is the portfolio artifact** — it shows you can build, fairly compare, and *reason about* the whole spectrum from a 2011 lexicon to a 2024 quantized LLM.

## 11. Recap — the whole arc

Eight notebooks, one benchmark. You built domain adaptation (Part 1), fine-tuning small and large (Parts 2–4), a no-training LLM baseline (Part 5), synthetic data (Part 6), and structured extraction (Part 7) — then scored them all, fairly, in one place. The lasting lessons:
- **Change one variable** — every comparison isolated a single cause.
- **Measure, don't assume** — memory, parameters, accuracy, cost: you instrumented each.
- **The cheapest method that clears the bar wins** — accuracy is necessary, not sufficient.

From here, the project's larger ambition (real 8-K EX-99.1 filings, earnings-surprise + soft-information regressions) is in `PROJECT_SCOPE.md` — and your extraction code from Part 7 is the on-ramp.